# MihocAlexT.ipynb - TensorFlow Solution

In [19]:
import os
import pickle as pkl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import recall_score, classification_report, confusion_matrix

model_save_path = "tf_mlp_merge_conflict_model.keras"

### 🔧 Data Preprocessing

- Dropped non-predictive identifiers from the dataset.
- Filled missing values using the **median** of each numeric column.
- Scaled all features using `StandardScaler` to help neural network convergence.

In [4]:
df = pd.read_csv('MergeConflictsDataset.csv', sep=';')
df.drop(columns=['commit', 'parent1', 'parent2', 'ancestor'], inplace=True)

if df.isnull().values.any():
    df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=['conflict'])
y = df['conflict']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 🧪 Train/Test Split

Performed a stratified train/test split to preserve class imbalance ratio in both sets. This is crucial for reliable model evaluation.


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

### ⚖️ Handling Class Imbalance

Instead of SMOTE, we used **class weights** to inform the model that class `1` (conflict) is underrepresented. This biases the model to better recall minority samples without altering the data.

In [10]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights to penalize imbalance during training
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {0: weights[0], 1: weights[1]}

### 🧠 Model Definition

- A simple **feedforward neural network** with dropout regularization to prevent overfitting.
- Monitors **validation recall** to optimize for our goal.
- Uses **binary cross-entropy** with sigmoid activation for binary classification.


In [18]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=[tf.keras.metrics.Recall()])

early_stopping = EarlyStopping(
    monitor='val_recall', mode='max', patience=5, restore_best_weights=True)

### 🏋️ Model Training and Saving

We train the model **only if a saved version is not already available**, allowing us to avoid re-training and reuse the best performing model.

- **Validation Recall** is monitored to apply **early stopping**.
- After training, the model is saved to disk using `model.save()`, making it easy to load and evaluate later.
- This approach ensures faster iteration and reproducibility, especially when training is time-consuming.

📁 Saved model path: `tf_mlp_merge_conflict_model.keras`


In [21]:
if os.path.exists(model_save_path):
    print("🔁 Loading saved TensorFlow model...")
    model = tf.keras.models.load_model(model_save_path)
else:
    print("🚀 Training new TensorFlow model...")
    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=35,
        batch_size=32,
        class_weight=class_weights_dict,
        callbacks=[early_stopping],
        verbose=1
    )
    # Save model
    model.save(model_save_path)
    print("✅ Model saved to:", model_save_path)

🚀 Training new TensorFlow model...
Epoch 1/35
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 457us/step - loss: 0.1119 - recall_3: 0.9853 - val_loss: 0.1561 - val_recall_3: 0.9559
Epoch 2/35
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 431us/step - loss: 0.1192 - recall_3: 0.9847 - val_loss: 0.1441 - val_recall_3: 0.9427
Epoch 3/35
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 424us/step - loss: 0.1048 - recall_3: 0.9920 - val_loss: 0.1605 - val_recall_3: 0.9648
Epoch 4/35
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 424us/step - loss: 0.1083 - recall_3: 0.9859 - val_loss: 0.1664 - val_recall_3: 0.9471
Epoch 5/35
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 429us/step - loss: 0.1081 - recall_3: 0.9930 - val_loss: 0.1602 - val_recall_3: 0.9339
Epoch 6/35
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 440us/step - loss: 0.1137 - recall_3: 0.9794 - val_loss: 0.1427 - val_recall_3: 0.9515
Epoch 7/35
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 422us/step - loss: 0.1094 - recall_3: 0.9850 - val_loss: 0.1754 - val_recall_3: 0.9559
Epoch 8/35
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step -

### 📊 Evaluation and Results

| Metric         | Non-Conflict (0) | Conflict (1) |
|----------------|------------------|--------------|
| Precision      | 1.00             | 0.52         |
| Recall         | 0.95             | **0.95**     |
| F1-score       | 0.97             | 0.67         |
| Support        | 5101             | 294          |

- **Recall for the `conflict` class (1)**: 0.945 → ✅ very closely aligned with our goal of maximizing recall.
- Only **16 actual conflict cases were missed** (false negatives).
- Excellent **generalization performance** due to early stopping, class weighting, and clean preprocessing.

🎯 **Conclusion**: The TensorFlow model strongly satisfies the recall maximization requirement for names A–K, and even slightly outperforms the Scikit-learn model in this regard.


In [26]:
# Predict
y_pred_probs = model.predict(X_test)
y_pred_tf = (y_pred_probs > 0.5).astype(int).flatten()

# Evaluate
print("\n🧠 TensorFlow Neural Network Results (No SMOTE)")
print("Recall:", recall_score(y_test, y_pred_tf))
print(confusion_matrix(y_test, y_pred_tf))
print(classification_report(y_test, y_pred_tf))


169/169 ━━━━━━━━━━━━━━━━━━━━ 0s 252us/step

🧠 TensorFlow Neural Network Results (No SMOTE)
Recall: 0.9455782312925171
[[4842  259]
 [  16  278]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      5101
           1       0.52      0.95      0.67       294

    accuracy                           0.95      5395
   macro avg       0.76      0.95      0.82      5395
weighted avg       0.97      0.95      0.96      5395



### ❗ Why Is Precision Low for the `conflict` Class?

While the model achieves **very high recall** for the `conflict` class (0.95), the **precision is notably lower (0.52)**. Here's why:

#### 📉 What Low Precision Means
Precision is defined as:

> **Precision = TP / (TP + FP)**  
> = proportion of predicted conflicts that were actually conflicts

A precision of 0.52 means that **about half of the samples predicted as "conflict" were actually not conflicts** (false positives).

#### 🔍 Why It Happens
1. **Class Imbalance**  
   The dataset is heavily imbalanced (many more "non-conflict" than "conflict" samples). To prioritize recall, the model becomes **more aggressive** in predicting "conflict" cases, increasing the number of **false positives** — which lowers precision.

2. **Class Weighting Bias**  
   We intentionally applied `class_weight='balanced'` to **boost the importance of minority class examples**. This pushes the model to prefer **detecting all true conflicts**, even at the cost of misclassifying some non-conflicts as conflicts.

3. **High Recall ↔ Lower Precision Trade-off**  
   In classification, **recall and precision are often in tension**. Maximizing one can lower the other. Since our goal (based on first name rule) is to **maximize recall**, we accept lower precision as a trade-off.

#### ✅ Why This Is Acceptable
In many real-world scenarios (like merge conflict prediction), **missing a conflict (false negative)** is far more costly than **flagging a false one (false positive)**.  
Thus, **high recall with tolerable precision is an acceptable and even desirable outcome** in this task.

